In [ ]:
!pip install -U dashscope

In [1]:
import os
import json
import re
import glob

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import TensorDataset, random_split

import ast
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, f1_score
import numpy as np


# Utility functions
def read_json(path):
    with open(path, 'r') as f:
        return json.load(f)

def get_training_weight(train_loader):
    def get_loader_stats(loader, name):
        weight_out=[]
        
        label_counts = {}
        feature_shapes = {}
        total_samples = 0

        for batch in loader:
            features, labels = batch
            total_samples += len(labels)

            for feature, label in zip(features, labels):
                label_val = label.item()
                label_counts[label_val] = label_counts.get(label_val, 0) + 1

                shape = tuple(feature.shape)
                feature_shapes[shape] = feature_shapes.get(shape, 0) + 1
        
        for label, count in sorted(label_counts.items()):
            weight_out.append(count)
            #print(f"    Label {label}: {count} samples")
        return weight_out
    return get_loader_stats(train_loader, "Training")    

def print_loader_metadata(train_loader, test_loader):
    def get_loader_stats(loader, name):
        label_counts = {}
        feature_shapes = {}
        total_samples = 0

        for batch in loader:
            features, labels = batch
            total_samples += len(labels)

            for feature, label in zip(features, labels):
                label_val = label.item()
                label_counts[label_val] = label_counts.get(label_val, 0) + 1

                shape = tuple(feature.shape)
                feature_shapes[shape] = feature_shapes.get(shape, 0) + 1

        print(f"\n{name} Loader:")
        print(f"  Total samples: {total_samples}")
        print(f"  Total batches (batch_size={loader.batch_size}): {len(loader)}")
        print(f"  Feature shape(s):")
        for shape, count in feature_shapes.items():
            print(f"    Shape {shape}: {count} samples")
        print(f"  Label distribution:")
        for label, count in sorted(label_counts.items()):
            print(f"    Label {label}: {count} samples")

    get_loader_stats(train_loader, "Training")
    get_loader_stats(test_loader, "Testing")

def get_final_D_cutoff_seconds(meta_for_video):
    """
    Parse the final D(mm:ss) from the 'timesec' string.
    Returns the cutoff in integer seconds, or None if there is no D.
    """
    if not meta_for_video:
        return None
    ts = meta_for_video.get('timesec', '')
    # Find ALL occurrences of D(mm:ss); use the last one as the cutoff.
    matches = list(re.finditer(r'D\((\d+):(\d+)\)', ts))
    if not matches:
        return None
    m = matches[-1]
    minutes = int(m.group(1))
    seconds = int(m.group(2))
    return minutes * 60 + seconds  # total seconds


def parse_augmented_filename(fname):
    """
    Parse names like '99A_0056_X.pt' into (base_id, timestamp_seconds, tag_suffix).
    - If timestamp token is 4 digits, interpret as MMSS.
    - Else if it's an integer string, interpret as total seconds.
    Returns (base_id, timestamp_seconds or None, tag_suffix or None).
    """
    stem = os.path.splitext(os.path.basename(fname))[0]  # e.g., '99A_0056_X'
    parts = stem.split('_')
    base = parts[0] if parts else None
    ts_seconds = None
    tag = parts[-1] if len(parts) >= 3 else (parts[1] if len(parts) == 2 else None)

    if len(parts) >= 2:
        ts_token = parts[1]
        if ts_token.isdigit():
            if len(ts_token) == 4:        # MMSS
                mm = int(ts_token[:2])
                ss = int(ts_token[2:])
                ts_seconds = mm * 60 + ss
            else:                          # total seconds
                ts_seconds = int(ts_token)

    return base, ts_seconds, tag


In [3]:
#prepare the data

#load the data from: 1- 00SF, 2- 01SF_sw

features_dir = '/Users/kesiyun/Desktop/00workspace/features/01SF_sw'
#features_dir = '/Users/kesiyun/Desktop/00workspace/features/01SF_sw_segcrop'
#features_dir = '/Users/kesiyun/Desktop/00workspace/features/01SF_sw_segobj'

split_path = '/Users/kesiyun/Desktop/00workspace/00full_video_split_5s.json'

loaded_result=read_json('/Users/kesiyun/Desktop/00workspace/00sequenceresult.json')
dataset_split=read_json(split_path)

training_split=dataset_split['training_set']
test_split=dataset_split['testing_set']
#print(training_split)
print(test_split)


# -------------------------
# Build TEST tensors
# -------------------------
test_features = []
test_labels   = []
skip_test=[]
#skip_test = ['16A']
 
for video_name in test_split:
    cutoff_seconds='n/a'
    if video_name in skip_test: continue

    print(video_name, 'added to testing (cutoff:', cutoff_seconds, ')')



['16A', '4C', '4A', '4E', '12B', '17I']
16A added to testing (cutoff: n/a )
4C added to testing (cutoff: n/a )
4A added to testing (cutoff: n/a )
4E added to testing (cutoff: n/a )
12B added to testing (cutoff: n/a )
17I added to testing (cutoff: n/a )


In [9]:
import os
import subprocess
from pathlib import Path


def get_video_duration(input_path):
    """Get video duration (in seconds) using ffprobe."""
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        input_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return float(result.stdout.strip())


def split_video_to_clips(
    input_video,
    output_root,
    clip_length=5,
    step=1,
    output_ext="mp4"
):
    input_video = str(input_video)
    output_root = Path(output_root)

    # Video name (no extension)
    video_name = Path(input_video).stem

    # Output folder
    output_dir = output_root / video_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Duration
    duration = get_video_duration(input_video)

    start = 0

    # Only full clips allowed → tail ignored automatically
    while start + clip_length <= duration:
        # ✅ filename = actual start time
        output_file = output_dir / f"{int(start)}.{output_ext}"

        # ✅ resume support
        if output_file.exists():
            print(f"Skipping existing: {output_file}")
            start += step
            continue

        cmd = [
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-i", input_video,
            "-t", str(clip_length),
            "-c", "copy",
            str(output_file)
        ]

        print(f"Creating: {output_file} ({start:.2f}s → {start+clip_length:.2f}s)")
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        start += step

    print("Done.")

In [11]:
for video_name in test_split:
    split_video_to_clips(
        '/Volumes/Green SSD/00Workspace_portable/videos/'+video_name+'.avi',
        "/Volumes/Green SSD/00Workspace_portable/test_simulation_videos",
        clip_length=5,
        step=5
    )

Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/0.mp4 (0.00s → 5.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/5.mp4 (5.00s → 10.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/10.mp4 (10.00s → 15.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/15.mp4 (15.00s → 20.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/20.mp4 (20.00s → 25.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/25.mp4 (25.00s → 30.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/30.mp4 (30.00s → 35.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/35.mp4 (35.00s → 40.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/40.mp4 (40.00s → 45.00s)
Creating: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A/45.mp4 (45

In [3]:
#https://billing-cost.console.aliyun.com/resource/spn/detail

#test: using strictly same prompt to ollama
#user only format -switchable
import os
import shutil
import pandas as pd
import dashscope

# --- CONFIGURATION ---

DIR_base = '/Volumes/Green SSD/00Workspace_portable/test_simulation_videos'

CSV_base = '/Users/kesiyun/Desktop/00workspace/bots/simu_csvs'

labels = {
    '16A': '16A', #'contraction': 'c',
    '4C': '4C',
    '4A': '4A',
    '4E': '4E',
    '12B': '12B',
    '17I': '17I'
}

DIR_list = [
    [
        os.path.join(DIR_base, k), 
        os.path.join(CSV_base, f'{v}_stentor_analysis_results.csv')
    ] 
    for k, v in labels.items()
]


for i in range (len(DIR_list)):
    INPUT_DIR = DIR_list[i][0]
    OUTPUT_CSV =DIR_list[i][1]
    #print(INPUT_DIR)
    #print(OUTPUT_CSV)


    #MODEL_NAME = 'qwen3-vl-235b-a22b-thinking'
    #MODEL_NAME = 'qwen3-vl-235b-a22b-instruct'
    MODEL_NAME = 'qwen3-vl-32b-thinking'

    ROLE_MODE = 'system-user'
    #ROLE_MODE = 'user-only' #'ollama'

    FPS_RATE = 7
    
    # --- PROMPT ---

    #ollama
    ollama_prompt_text = r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no” (such as “maybe,” “I’m not sure,” etc.). In addition to your main answer, give a brief explanation for why you did or did not observe a contraction in the video. 
    
    meanwhile please provide an integer contraction score for your judgement, where 0 for not a contraction and 100 for a sure contraction, so a score in the middle like 50 represents a hard case.
    response format: "[yes or no]/[contraction score]/[reasoning]" e.g. "yes/87/At 0.2 s, the stentor is fully......... Therefore, contraction occurs....."
    warning: must response in english regardless of my location, timezone and system/browser language settings. 
    
    When classifying stentor behaviors, keep in mind the following important points:
    -) Most contractions happen quickly (often taking just one subsampled frame). Some contractions, however, are slower and may take ten or more frames. “Slow” contractions should still be annotated as contractions. 
    -) Not all contractions result in a fully ball-like shape. Some contractions involve a clear and measurable shortening of the stentor’s body length, where the distance between the “head” and “tail” decreases noticeably but the organism does not become spherical. These partial or incomplete contractions should still be annotated as contractions. 
    -) The video may show objects other than the stentor, such as algae, plastic beads, glass needles, or similar debris. You should ignore these objects and focus only on the stentor. 
    Besides, the coming video associated with this task should be evaluated independently based solely on its own visual evidence. If additional videos follow, each video should be treated as a separate and independent task. Do not use information, observations, or conclusions from previous videos when evaluating later videos."""

    API_system_prompt_text=  r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    
    Meanwhile please provide an integer contraction score for your judgement, where 0 = not a contraction and 100 = sure contraction.
    Response format:
    "[yes or no]/[contraction score]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """

    no_score_API_system_prompt_text=  r"""For this task, you are an expert at identifying the behaviors of stentors, a type of unicellular ciliated protist. 
    The uploaded video shows a stentor under magnification, which may be engaging in one or more typical behaviors. 
    The video consists of subsampled frames. Given the evidence in the video, identify whether the stentor “contracts.” 
    The definition of a “contraction” is any behavior in which the stentor changes from an extended shape resembling a trumpet to a condensed, ball-like shape. 
    
    Answer “yes” if the stentor contracts at any point in the video; answer “no” if it does not. Do not provide any answer other than “yes” or “no”. 
    In addition to your main answer, give a brief explanation.
    Response format:
    "[yes or no]/[reasoning]"
    
    Important notes:
    - Slow contractions still count
    - Partial shortening still counts
    - Ignore debris and non-stentor objects
    - Each video must be classified independently
    - Always respond in English.
    """

    

    #Ollama: user-only
    #prompt_text = ollama_prompt_text #original prompt used on ollama (ubuntu)
    prompt_text = API_system_prompt_text #to test the difference caused by modes
    
    #API_SYSTEM
    SYSTEM_PROMPT = no_score_API_system_prompt_text #API_system_prompt_text

    #API_SYSTEM
    USER_PROMPT =r"""Analyze this video."""
    
    # --- MAIN PIPELINE ---
    
    # 1. Load processed records
    processed_files = set()
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        processed_files = set(df_existing['filename'].tolist())
    
    # 2. List videos
    valid_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    all_videos = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]
    
    print(f"Found {len(all_videos)} videos. {len(processed_files)} already processed.")
    print("model:", MODEL_NAME)
    print("dir:", INPUT_DIR)
    print("mode:",ROLE_MODE)

    process_index=0
    
    for video_name in all_videos:
        process_index+=1
        if video_name in processed_files:
            print(f"Skipping: {video_name}")
            continue
    
        video_path = os.path.join(INPUT_DIR, video_name)
        print(f"\nProcessing: {video_name}...")
    
        try:
            video_uri = f"file://{os.path.abspath(video_path)}"

            if ROLE_MODE=='user-only': 
                messages = [{
                        "role": "user",
                        "content": [
                            {"text": prompt_text},
                            {"video": video_uri, "fps": FPS_RATE},
                        ]
                    }
                ]
    

            else: #ROLE_MODE=='system-user': 
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {
                        "role": "user",
                        "content": [
                            {"video": video_uri, "fps": FPS_RATE},
                            {"text": USER_PROMPT}
                        ]
                    }
                ]
        
            response = dashscope.MultiModalConversation.call(
                api_key=os.getenv('DASHSCOPE_API_KEY'),
                model=MODEL_NAME,
                messages=messages
            )
    
            # --- STATUS CHECK (NEW) ---
            if response.status_code != 200:
                print(f"❌ HTTP status: {response.status_code}")
                print(f"Error code: {getattr(response, 'code', 'N/A')}")
                print(f"Error message: {getattr(response, 'message', 'N/A')}")
                continue
    
            # --- PARSE RESPONSE ---
            ai_response = response.output.choices[0].message.content[0]["text"]
    
            # --- TOKEN USAGE PRINT (NEW, SAFE ACCESS) ---
            usage = getattr(response, "usage", None)
            if usage is not None:
                input_tokens = getattr(usage, "input_tokens", None)
                output_tokens = getattr(usage, "output_tokens", None)
                total_tokens = getattr(usage, "total_tokens", None)
                print(
                    f"✅ Token usage — "
                    f"input: {input_tokens}, output: {output_tokens}, total: {total_tokens}"
                )
            else:
                print("✅ Processed successfully (token usage not returned).")
    
            new_row = {
                'filename': video_name,
                'response': ai_response
            }
    
            # Append immediately (crash-safe)
            pd.DataFrame([new_row]).to_csv(
                OUTPUT_CSV,
                mode='a',
                index=False,
                header=not os.path.exists(OUTPUT_CSV)
            )
    
            print(f"Done: {video_name}, {process_index}/{len(all_videos)}")
    
        except Exception as e:
            print(f"❌ Exception processing {video_name}: {e}")
    
    print(f"\nAll done! Results saved to {OUTPUT_CSV}\n")

Found 466 videos. 466 already processed.
model: qwen3-vl-32b-thinking
dir: /Volumes/Green SSD/00Workspace_portable/test_simulation_videos/16A
mode: system-user
Skipping: 550.mp4
Skipping: 2240.mp4
Skipping: 1985.mp4
Skipping: 1775.mp4
Skipping: 2095.mp4
Skipping: 975.mp4
Skipping: 785.mp4
Skipping: 1205.mp4
Skipping: 140.mp4
Skipping: 815.mp4
Skipping: 55.mp4
Skipping: 1365.mp4
Skipping: 430.mp4
Skipping: 2320.mp4
Skipping: 395.mp4
Skipping: 1615.mp4
Skipping: 380.mp4
Skipping: 1600.mp4
Skipping: 425.mp4
Skipping: 1370.mp4
Skipping: 40.mp4
Skipping: 800.mp4
Skipping: 155.mp4
Skipping: 790.mp4
Skipping: 1210.mp4
Skipping: 960.mp4
Skipping: 2080.mp4
Skipping: 1760.mp4
Skipping: 2255.mp4
Skipping: 1990.mp4
Skipping: 545.mp4
Skipping: 1945.mp4
Skipping: 2280.mp4
Skipping: 235.mp4
Skipping: 590.mp4
Skipping: 1010.mp4
Skipping: 745.mp4
Skipping: 2055.mp4
Skipping: 1560.mp4
Skipping: 95.mp4
Skipping: 625.mp4
Skipping: 2135.mp4
Skipping: 180.mp4
Skipping: 1400.mp4
Skipping: 1825.mp4
Skipping: 

In [7]:
import os
import re
import pandas as pd
from collections import defaultdict

#'16A': 466 ~$12.79
#'4C': 21 ~$0.58
#'4A': 191 ~$5.24
#'4E': 120 ~$3.29
#'12B': 746 ~$20.47
#'17I': 96 ~$2.64

#total: 1640 ~$40~~45,

# =========================
# Ground truth dictionary
# =========================
GROUND_TRUTH = {
    '16A': 'R P(1:45) A(1:58)B(2:08)C(5:10)C(15:38)C(21:39)C(26:34) P(36:03) C(36:30)D(38:45)',
    '4C': 'R P(0:15) A(0:18)C(0:19)',
    '4A': 'R P(0:16) P(0:46) C(0:55) P(14:14) R P(0:16) P(0:46) C(1:14) P(14:14) R P(0:16) P(0:46) C(1:21) P(14:14)',
    '4E': 'R P(0:58) A(1:01)C(1:02)C(1:45) P(5:02) D(5:22)',
    '12B': 'R P(0:30) C(0:31) P(5:16) B(5:20)C(5:23)B(10:21) P(11:37) C(11:42)B(16:06) P(18:04) C(18:06) P(28:02) B(28:07)C(28:08) P(46:45) C(46:56)D(47:55) R P(50:00) C(50:03)',
    '17I': 'R P(0:33) A(0:59)B(1:21)C(1:26)C(3:40)C(4:18)C(4:57)D(5:18)',
}

INTERVAL = 5  # seconds


# =========================
# Helpers
# =========================
def time_to_seconds(t):
    """Convert mm:ss to seconds."""
    m, s = map(int, t.split(":"))
    return m * 60 + s


def extract_positive_intervals(gt_string):
    """
    Extract all intervals that contain at least one C(timestamp).
    Return set of starting seconds (e.g., 0, 5, 10...).
    """
    matches = re.findall(r'C\((\d+:\d+)\)', gt_string)
    intervals = set()

    for t in matches:
        sec = time_to_seconds(t)
        start = (sec // INTERVAL) * INTERVAL
        intervals.add(start)

    return intervals


def parse_prediction(resp):
    """Extract yes/no (robust to case, formatting)."""
    if pd.isna(resp):
        return None
    text = str(resp).strip().lower()
    if text.startswith("yes"):
        return 1
    elif text.startswith("no"):
        return 0
    return None


# =========================
# Main Evaluation Function
# =========================
def evaluate_directory(input_dir):
    TP_c = FP_c = FN_c = TN_c = 0

    # Precompute ground truth intervals
    gt_intervals = {
        key: extract_positive_intervals(val)
        for key, val in GROUND_TRUTH.items()
    }

    for file in os.listdir(input_dir):
        if not file.endswith("_stentor_analysis_results.csv"):
            continue

        # Extract video key (e.g., 16A)
        video_key = file.split("_")[0]

        if video_key not in gt_intervals:
            continue

        gt_pos = gt_intervals[video_key]

        df = pd.read_csv(os.path.join(input_dir, file))

        for _, row in df.iterrows():
            fname = row["filename"]
            pred = parse_prediction(row["response"])

            if pred is None:
                continue

            try:
                sec = int(fname.replace(".mp4", ""))
            except:
                continue

            is_pos = sec in gt_pos

            if pred == 1 and is_pos:
                TP_c += 1
            elif pred == 1 and not is_pos:
                FP_c += 1
            elif pred == 0 and is_pos:
                FN_c += 1
            elif pred == 0 and not is_pos:
                TN_c += 1

    # =========================
    # Metrics
    # =========================
    precision_C = TP_c / (TP_c + FP_c) if (TP_c + FP_c) > 0 else 0
    recall_C = TP_c / (TP_c + FN_c) if (TP_c + FN_c) > 0 else 0
    f1_C = (2 * precision_C * recall_C / (precision_C + recall_C)) if (precision_C + recall_C) > 0 else 0
    accuracy = (TP_c + TN_c) / (TP_c + TN_c + FP_c + FN_c) if (TP_c + TN_c + FP_c + FN_c) > 0 else 0

    # Macro (binary: C vs not-C)
    precision_notC = TN_c / (TN_c + FN_c) if (TN_c + FN_c) > 0 else 0
    recall_notC = TN_c / (TN_c + FP_c) if (TN_c + FP_c) > 0 else 0
    f1_notC = (2 * precision_notC * recall_notC / (precision_notC + recall_notC)) if (precision_notC + recall_notC) > 0 else 0

    macro_precision = (precision_C + precision_notC) / 2
    macro_f1 = (f1_C + f1_notC) / 2

    # =========================
    # Print Results
    # =========================
    print("\n--- Detailed Evaluation on Test Set ---")
    print(f"True Positives (C): {TP_c*5}")
    print(f"False Positives (C): {FP_c*5}")
    print(f"False Negatives (C): {FN_c*5}")
    print(f"Precision (C): {precision_C:.4f}")
    print(f"Recall (C): {recall_C:.4f}")
    print(f"F1 Score (C): {f1_C:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro Precision: {macro_precision:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")



In [8]:
input_dir = "/Users/kesiyun/Desktop/00workspace/bots/simu_csvs/clip5step5_235b"
evaluate_directory(input_dir)


--- Detailed Evaluation on Test Set ---
True Positives (C): 40
False Positives (C): 330
False Negatives (C): 70
Precision (C): 0.1081
Recall (C): 0.3636
F1 Score (C): 0.1667
Accuracy: 0.9512
Macro Precision: 0.5496
Macro F1: 0.5708


In [9]:
input_dir = "/Users/kesiyun/Desktop/00workspace/bots/simu_csvs/clip5step5_32b"
evaluate_directory(input_dir)


--- Detailed Evaluation on Test Set ---
True Positives (C): 60
False Positives (C): 605
False Negatives (C): 50
Precision (C): 0.0902
Recall (C): 0.5455
F1 Score (C): 0.1548
Accuracy: 0.9201
Macro Precision: 0.5418
Macro F1: 0.5564
